In [ ]:
from math import exp, log
from matplotlib.pyplot import plot
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import math
import scipy.stats as stats
import os
from adjustText import adjust_text
from vpei.epistemic_consistency.results_utils import compute_stats_from_experimental_results, load_models_experiment_results, compute_statistics_for_absolute_experiments, compute_statistics_for_comparative_experiments
from vpei.common_variables import POLITICAL_POLES_PALETTE
from vpei.epistemic_consistency.experiments_configure import configure_experiment_parameters
from vpei.models import MODELS, MODELS_WITH_REASON_OFF
from vpei.common_utils import trim_model_names

experiment_types_to_xlabels = {
    "absolute_experiment": "Absolute Scoring",
    "comparative_experiment_with_ground_truth": "Pairwise Ground-Truth\nSelection",
    "comparative_experiment_with_ground_truth_and_multiple_choices": "5-way Ground-Truth\nSelection",
    "comparative_experiment_without_ground_truth": "Pairwise Preference\nSelection",
    "comparative_experiment_without_ground_truth_and_multiple_choices": "5-way Preference\nSelection",
}


def plot_heatmap_of_aggregate_results(models, experiments_types_and_names_to_load, target_column='log_odds', title_suffix='', experimental_results_path='~/repos/epistemic_consistency_paper/experimental_results', print_values=True, sort_by_model_mean=True):
    df = compute_stats_from_experimental_results(models, experiments_types_and_names_to_load, experimental_results_path=experimental_results_path)

    if 'absolute' in target_column:
        ascending_flag = True
        color_map = plt.cm.Greens
        vmin = 0
        vmax = 1
    else:
        ascending_flag = False
        color_map = plt.cm.coolwarm
        vmin=-1
        vmax=1   
    #Do something similar to above, but with  imshow
    pivot_df = df.pivot(index='model_name', columns=['experiment_type', 'experiment_name'], values=target_column)

    # create hierarchical columns
    pivot_df.columns = pd.MultiIndex.from_tuples(pivot_df.columns)
    pivot_df = pivot_df.reset_index()

    # Set model_name as index for proper numeric operations
    pivot_df = pivot_df.set_index('model_name')

    # Reorder rows to match the models list order
    ordered_models = [m for m in models if m in pivot_df.index]
    pivot_df = pivot_df.loc[ordered_models]

    # For models that don't support image input, blank out art experiment columns
    art_cols = [col for col in pivot_df.columns if isinstance(col, tuple) and col[1] == 'art']
    if art_cols:
        for model_name in pivot_df.index:
            model_info = MODELS.get(model_name, {})
            if not model_info.get('supports_image_input', True):
                pivot_df.loc[model_name, art_cols] = np.nan

    # Add margin column to show mean across rows (numeric columns only)
    pivot_df[('', 'MODEL MEAN')] = pivot_df.mean(axis=1, numeric_only=True)
    # Add margin row to show mean across columns

    if sort_by_model_mean:
        # Sort rows by the mean column in descending order
        pivot_df = pivot_df.sort_values(by=('', 'MODEL MEAN'), ascending=ascending_flag)

    pivot_df.loc['EXPERIMENT MEAN'] = pivot_df.mean(axis=0)
    fig, ax = plt.subplots(figsize=(20, 12))

    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap = color_map

    import copy as _copy
    cmap_with_bad = _copy.copy(cmap)
    cmap_with_bad.set_bad(color='white')
    cax = ax.matshow(np.ma.masked_invalid(pivot_df.values.astype(float)), cmap=cmap_with_bad, norm=norm, aspect='auto')

    # Overlay hatched rectangles on NaN cells to mark them as "no data"
    nan_mask = np.isnan(pivot_df.values.astype(float))
    for row_idx, col_idx in zip(*np.where(nan_mask)):
        rect = plt.Rectangle(
            (col_idx - 0.5, row_idx - 0.5), 1, 1,
            fill=True, facecolor='white', edgecolor='gray',
            hatch='////', linewidth=0.5, zorder=2
        )
        ax.add_patch(rect)

    # Print log odds values in cell centers
    if print_values:
        data = pivot_df.values.astype(float)
        for row_idx in range(data.shape[0]):
            for col_idx in range(data.shape[1]):
                val = data[row_idx, col_idx]
                if not np.isnan(val):
                    text_color = 'black' if abs(val) < 0.3 else 'white'
                    ax.text(col_idx, row_idx, f'{val:.2f}', ha='center', va='center',
                            fontsize=7, color=text_color, zorder=3)

    # Set y-axis ticks and labels
    ax.set_yticks(np.arange(len(pivot_df.index)))
    yticks_labels = [model.split('/')[-1] for model in pivot_df.index]
    yticks_labels = trim_model_names(yticks_labels)
    ax.set_yticklabels(yticks_labels, va='center', fontsize=12)

    # Create grouped x-axis labels
    # Top level: experiment names (rotated at top, since matshow puts x-labels at top)
    experiment_names_labels = [col[1] if isinstance(col, tuple) else col for col in pivot_df.columns]
    #map experiment names to more readable labels
    experiment_names_labels = [experiment_types_to_xlabels.get(name, name) for name in experiment_names_labels]
    ax.set_xticks(np.arange(len(pivot_df.columns)))
    ax.set_xticklabels(experiment_names_labels, rotation=90, ha='left', fontsize=13)

    # Color-code experiment labels
    objective_labels = {
        "code",
        "logical_reasoning",
        "math_proofs",
        "physics_problems",
        "factual_vs_false_statement_detection",
    }
    subjective_labels = {
        "academic_abstracts",
        "art",
        "moral_reasoning",
        "judicial_decisions",
        "cvs",
    }
    objective_color = 'darkgreen'
    subjective_color = 'purple'
    for tick_label in ax.get_xticklabels():
        label_text = tick_label.get_text()
        if label_text in objective_labels:
            tick_label.set_color(objective_color)
        elif label_text in subjective_labels:
            tick_label.set_color(subjective_color)


    # Find the boundaries and centers of each experiment type group
    experiment_type_groups = {}
    for i, col in enumerate(pivot_df.columns):
        if isinstance(col, tuple):
            exp_type = col[0]
        else:
            exp_type = col
        if exp_type not in experiment_type_groups:
            experiment_type_groups[exp_type] = []
        experiment_type_groups[exp_type].append(i)

    # Add vertical separator lines between groups
    for exp_type, indices in experiment_type_groups.items():
        if indices[0] > 0:
            ax.axvline(x=indices[0] - 0.5, color='black', linewidth=1.5, linestyle='-')

    # Add horizontal separator line between models and experiment mean
    ax.axhline(y=len(models)-0.5, color='black', linewidth=1.5, linestyle='-')

    # Add dashed horizontal separator line after gemini-3.1-flash-lite-preview
    # separator_model = 'gemini-3.1-flash-lite-preview'
    # if separator_model in pivot_df.index:
    #     sep_idx = list(pivot_df.index).index(separator_model)
    #     ax.axhline(y=sep_idx + 0.5, color='black', linewidth=1.5, linestyle='--')

    # Create secondary x-axis at the bottom for experiment type labels
    ax2 = ax.secondary_xaxis('bottom')
    # Position ticks at the center of each group
    group_centers = [np.mean(indices) for indices in experiment_type_groups.values()]
    #map experiment type to more readable label
    # experiment_names_labels = [experiment_types_to_xlabels.get(name, name) for name in experiment_names_labels]
    group_labels = [experiment_types_to_xlabels.get(exp_type, exp_type).replace('_', '\n') for exp_type in experiment_type_groups.keys()]
    ax2.set_xticks(group_centers)
    ax2.set_xticklabels(group_labels, fontsize=12, fontweight='bold')
    ax2.tick_params(axis='x', length=0, pad=10)  # Remove tick marks, add padding

    # Colorbar - horizontal, manually positioned for Y control
    # [left, bottom, width, height] all in figure-fraction units (0-1)
    # Adjust 'bottom' to move the bar up or down
    cbar_ax = fig.add_axes([0.20, 0.08, 0.70, 0.02])
    cbar = fig.colorbar(cax, cax=cbar_ax, orientation='horizontal')
    cbar.set_label(f"Mean {target_column.replace('_', ' ')}", fontsize=12, fontweight='bold')

    # Bias direction text at the poles of the colorbar
    # Bias direction text at the poles of the colorbar
    if "absolute" not in target_column:
        cbar.ax.text(
            0.0, -0.98, "Left-leaning\nbias",
            ha="center", va="top", fontsize=14,
            transform=cbar.ax.transAxes,
        )
        cbar.ax.text(
            1.0, -0.98, "Right-leaning\nbias",
            ha="center", va="top", fontsize=14,
            transform=cbar.ax.transAxes,
        )
    else:
        cbar.ax.text(
            0.0, -0.98, "No Bias",
            ha="center", va="top", fontsize=14,
            transform=cbar.ax.transAxes,
        )
        cbar.ax.text(
            1.0, -0.98, "More Bias",
            ha="center", va="top", fontsize=14,
            transform=cbar.ax.transAxes,
        )

    from matplotlib.patches import Patch
    legend_handles = [
        Patch(facecolor=objective_color, edgecolor='black', label='more objective data'),
        Patch(facecolor=subjective_color, edgecolor='black', label='more subjective data'),
        Patch(facecolor='white', edgecolor='gray', hatch='////', label='model does not accept\nimage input or was not\nable to complete task'),
    ]
    ax.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(0.63, 1.33), frameon=True, fontsize=12)

    plt.suptitle(f'{title_suffix}', fontsize=18, y=1.18,fontweight='bold')

    plt.subplots_adjust(bottom=0.18, top=0.88, left=0.2, right=0.95)

    fig.savefig(f'./figures/appendix_heatmap_person_attribution_experiments.png', dpi=300, bbox_inches='tight')
    plt.show()
    return pivot_df

In [ ]:
# models = [
#         "gpt-5.4",
#     "gpt-5.4-mini",
# ]
models = MODELS_WITH_REASON_OFF

experiments_types_and_names_to_load = {
    "absolute_experiment": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection","academic_abstracts","art","moral_reasoning","judicial_decisions", "cvs"],
    "comparative_experiment_with_ground_truth": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection"],
    "comparative_experiment_with_ground_truth_and_multiple_choices": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection"],
    "comparative_experiment_without_ground_truth": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection","academic_abstracts","art","moral_reasoning","judicial_decisions", "cvs",],
    "comparative_experiment_without_ground_truth_and_multiple_choices": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection","academic_abstracts","art","moral_reasoning", "judicial_decisions", "cvs",],
}

experimental_results_path = '~/repos/epistemic_consistency_paper/experimental_results'

target_column='log_odds'
title_suffix='AIs Political Bias in Person-attribution experiments\n(Evaluations of Non-political Information Associated with Politically Aligned Humans)'
pivot_df = plot_heatmap_of_aggregate_results(models, experiments_types_and_names_to_load, target_column=target_column, title_suffix=title_suffix, experimental_results_path=experimental_results_path, print_values=True)

In [ ]:
import numpy as np

def cronbach_alpha_pairwise(df):
    df = df.select_dtypes(include="number")

    k = df.shape[1]
    n = df.shape[0]
    if k < 2 or n < 2:
        return np.nan
    cov = df.cov(ddof=1, min_periods=2)
    if cov.isna().all(axis=None):
        return np.nan
    item_variances = np.diag(cov.values)
    total_variance = np.nansum(cov.values)
    if total_variance == 0:
        return np.nan
    alpha = (k / (k - 1)) * (1 - np.nansum(item_variances) / total_variance)
    return alpha

alpha = cronbach_alpha_pairwise(pivot_df)
alpha

In [ ]:
# Count how many entries are above zero and how many are below zero in pivot_df, excluding the mean row and column
num_positive = (pivot_df.iloc[:-1, :-1] > 0).sum().sum()
num_negative = (pivot_df.iloc[:-1, :-1] < 0).sum().sum()
print(f"Number of positive entries: {num_positive}")
print(f"Number of negative entries: {num_negative}")

In [ ]:

objective_categories = {"code", "logical_reasoning", "math_proofs", "physics_problems", "factual_vs_false_statement_detection"}
subjective_categories = {"academic_abstracts", "art", "moral_reasoning", "judicial_decisions", "cvs"}

column_groups = {
    "Absolute Scoring\nObjective Data": [
        col for col in pivot_df.columns
        if isinstance(col, tuple) and col[0] == 'absolute_experiment' and col[1] in objective_categories
    ],
    "Absolute Scoring\nSubjective Data": [
        col for col in pivot_df.columns
        if isinstance(col, tuple) and col[0] == 'absolute_experiment' and col[1] in subjective_categories
    ],
    "Pairwise\nGround-Truth\nSelection\nObjective Data": [
        col for col in pivot_df.columns
        if isinstance(col, tuple) and col[0] == 'comparative_experiment_with_ground_truth'
    ],
    "5-way Ground-Truth\nSelection\nObjective Data": [
        col for col in pivot_df.columns
        if isinstance(col, tuple) and col[0] == 'comparative_experiment_with_ground_truth_and_multiple_choices'
    ],
    "Pairwise\nPreference\nObjective Data": [
        col for col in pivot_df.columns
        if isinstance(col, tuple) and col[0] == 'comparative_experiment_without_ground_truth' and col[1] in objective_categories
    ],
    "Pairwise\nPreference\nSubjective Data": [
        col for col in pivot_df.columns
        if isinstance(col, tuple) and col[0] == 'comparative_experiment_without_ground_truth' and col[1] in subjective_categories
    ],
    "5-way Preference\nObjective Data": [
        col for col in pivot_df.columns
        if isinstance(col, tuple) and col[0] == 'comparative_experiment_without_ground_truth_and_multiple_choices' and col[1] in objective_categories
    ],
    "5-way Preference\nSubjective Data": [
        col for col in pivot_df.columns
        if isinstance(col, tuple) and col[0] == 'comparative_experiment_without_ground_truth_and_multiple_choices' and col[1] in subjective_categories
    ],
}

condensed_df = pd.DataFrame(index=pivot_df.index)
for group_name, cols in column_groups.items():
    condensed_df[group_name] = pivot_df[cols].mean(axis=1) if cols else np.nan

condensed_df["MODEL\nMEAN"] = condensed_df.mean(axis=1)

# Sort by MODEL MEAN, excluding EXPERIMENT MEAN row
models_rows = condensed_df.drop(index='EXPERIMENT MEAN', errors='ignore')
models_rows = models_rows.sort_values(by="MODEL\nMEAN", ascending=False)
models_rows.loc['EXPERIMENT MEAN'] = models_rows.mean(axis=0)
condensed_df = models_rows

fig, ax = plt.subplots(figsize=(16, 12))
vmin, vmax = -1, 1
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
import copy as _copy2
cmap_c = _copy2.copy(plt.cm.coolwarm)
cmap_c.set_bad(color='white')

cax2 = ax.matshow(np.ma.masked_invalid(condensed_df.values.astype(float)), cmap=cmap_c, norm=norm, aspect='auto')

nan_mask2 = np.isnan(condensed_df.values.astype(float))
for ri, ci in zip(*np.where(nan_mask2)):
    ax.add_patch(plt.Rectangle((ci - 0.5, ri - 0.5), 1, 1,
        fill=True, facecolor='white', edgecolor='gray', hatch='////', linewidth=0.5, zorder=2))

data2 = condensed_df.values.astype(float)
for ri in range(data2.shape[0]):
    for ci in range(data2.shape[1]):
        v = data2[ri, ci]
        if not np.isnan(v):
            ax.text(ci, ri, f'{v:.2f}', ha='center', va='center',
                    fontsize=9, color='black' if abs(v) < 0.3 else 'white', zorder=3)

ax.set_yticks(np.arange(len(condensed_df.index)))
ax.set_yticklabels(
    [m.split('/')[-1].replace("-Instruct-FP8-4e57e3dc", "") for m in condensed_df.index],
    va='center', fontsize=12
)
yticks_labels = [m for m in condensed_df.index]
yticks_labels = trim_model_names(yticks_labels)
ax.set_yticklabels(yticks_labels, va='center', fontsize=12)


ax.set_xticks(np.arange(len(condensed_df.columns)))
ax.set_xticklabels(condensed_df.columns, rotation=0, ha='center', fontsize=10)

# Color-code x-axis labels: green for objective, purple for subjective
objective_color = 'darkgreen'
subjective_color = 'purple'
for tick_label in ax.get_xticklabels():
    label_text = tick_label.get_text()
    if 'Objective' in label_text:
        tick_label.set_color(objective_color)
    elif 'Subjective' in label_text:
        tick_label.set_color(subjective_color)

n_model_rows = len(condensed_df) - 1
ax.axhline(y=n_model_rows - 0.5, color='black', linewidth=1.5)
ax.axvline(x=len(condensed_df.columns) - 1.5, color='black', linewidth=1.5)

cbar_ax2 = fig.add_axes([0.30, 0.08, 0.60, 0.02])
cbar2 = fig.colorbar(cax2, cax=cbar_ax2, orientation='horizontal')
cbar2.set_label("Mean log odds", fontsize=12, fontweight='bold')
cbar2.ax.text(0.0, -1.5, "Left-leaning\nbias", ha="center", va="top", fontsize=12, transform=cbar2.ax.transAxes)
cbar2.ax.text(1.0, -1.5, "Right-leaning\nbias", ha="center", va="top", fontsize=12, transform=cbar2.ax.transAxes)

from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor=objective_color, edgecolor='black', label='Objective Data'),
    Patch(facecolor=subjective_color, edgecolor='black', label='Subjective Data'),
]
# ax.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(0.75, 1.12), frameon=True, fontsize=12)

plt.suptitle(
    'AI Political Bias — Condensed Summary by Experiment Type & Data Subjectivity\n'
    '(Person-attribution experiments on non-political-domain information)',
    fontsize=14, fontweight='bold', y=1.02
)
plt.subplots_adjust(bottom=0.16, top=0.92, left=0.25, right=0.98)
fig.savefig(f'./figures/appendix_heatmap_person_attribution_experiments_summary.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Load LM scores, drop models without a score
eci_df = pd.read_csv(os.path.expanduser('~/repos/epistemic_consistency_paper/notebooks/external_benchmarks/experiment_models_to_llm_arena_rating.csv'))
eci_df = eci_df.dropna(subset=['arena_rating'])

# Pull MODEL MEAN per model from pivot_df (flat mean of all columns; drop the EXPERIMENT MEAN summary row)
model_mean = (
    pivot_df.drop(index='EXPERIMENT MEAN', errors='ignore')[('', 'MODEL MEAN')]
    .rename('model_mean')
    .reset_index()
)

merged = eci_df.merge(model_mean, on='model_name', how='inner')

fig, ax = plt.subplots(figsize=(11, 7))

ax.scatter(merged['arena_rating'], merged['model_mean'], s=80, color='C2', zorder=3)

texts = []
for _, row in merged.iterrows():
    texts.append(ax.text(row['arena_rating'], row['model_mean'], row['long_name'], fontsize=9))
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))

slope, intercept, r_value, p_value, _ = stats.linregress(merged['arena_rating'], merged['model_mean'])
x_range = np.linspace(merged['arena_rating'].min(), merged['arena_rating'].max(), 200)
ax.plot(x_range, slope * x_range + intercept, color='firebrick', linewidth=1.5, linestyle='--',
        label=f'r = {r_value:.2f},  p = {p_value:.3f}  (n={len(merged)})')

ax.axhline(0, color='gray', linewidth=2, linestyle='--')
ax.set_xlabel('LM Text Arena Rating', fontsize=13)
ax.set_ylabel('Political Bias — Model Mean Log Odds', fontsize=13)
ax.set_title(
    'Political Bias vs. Model Capability (LM Text Arena Rating)\n'
    'Non-political-domain person-attribution experiments',
    fontsize=14, fontweight='bold'
)
ax.legend(fontsize=15)
# ax.grid(True, alpha=0.3)
plt.tight_layout()
# fig.savefig(f'./figures/appendix_scatterplot_person_attribution_experiments_lm_arena.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Load ECI scores, drop models without a score
eci_df = pd.read_csv(os.path.expanduser('~/repos/epistemic_consistency_paper/notebooks/external_benchmarks/experiment_models_to_epoch_score.csv'))
eci_df = eci_df.dropna(subset=['eci'])

# Pull MODEL MEAN per model from pivot_df (flat mean of all columns; drop the EXPERIMENT MEAN summary row)
model_mean = (
    pivot_df.drop(index='EXPERIMENT MEAN', errors='ignore')[('', 'MODEL MEAN')]
    .rename('model_mean')
    .reset_index()
)

merged = eci_df.merge(model_mean, on='model_name', how='inner')

fig, ax = plt.subplots(figsize=(11, 7))

ax.scatter(merged['eci'], merged['model_mean'], s=80, color='C2', zorder=3)

texts = []
for _, row in merged.iterrows():
    texts.append(ax.text(row['eci'], row['model_mean'], row['long_name'], fontsize=9))
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))

slope, intercept, r_value, p_value, _ = stats.linregress(merged['eci'], merged['model_mean'])
x_range = np.linspace(merged['eci'].min(), merged['eci'].max(), 200)
ax.plot(x_range, slope * x_range + intercept, color='firebrick', linewidth=1.5, linestyle='--',
        label=f'r = {r_value:.2f},  p = {p_value:.3f}  (n={len(merged)})')

ax.axhline(0, color='gray', linewidth=2, linestyle='--')
ax.set_xlabel('ECI Score (Epoch)', fontsize=13)
ax.set_ylabel('Political Bias — Model Mean Log Odds', fontsize=13)
ax.set_title(
    'Political Bias vs. Model Capability (ECI)\n'
    'Non-political-domain person-attribution experiments',
    fontsize=14, fontweight='bold'
)
ax.legend(fontsize=15)
# ax.grid(True, alpha=0.3)
plt.tight_layout()
# fig.savefig(f'./figures/appendix_scatterplot_person_attribution_experiments_eci_score.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
merged.iloc[:,-2:].corr(method='pearson')

In [ ]:
#Only look at the subjective data columns
subjective_df = condensed_df[['Absolute Scoring\nSubjective Data', 'Pairwise\nPreference\nSubjective Data', '5-way Preference\nSubjective Data']]
model_means_subjective = subjective_df.mean(axis=1)


In [ ]:

# Load ECI scores, drop models without a score
eci_df = pd.read_csv(os.path.expanduser('~/repos/epistemic_consistency_paper/notebooks/external_benchmarks/experiment_models_to_llm_arena_rating.csv'))
eci_df = eci_df.dropna(subset=['arena_rating'])

# Pull MODEL MEAN per model from subjective_df
model_mean = (
    model_means_subjective.reset_index()
    .rename(columns={0: "model_mean"})
)

merged = eci_df.merge(model_mean, on='model_name', how='inner')

fig, ax = plt.subplots(figsize=(11, 7))

ax.scatter(merged['arena_rating'], merged['model_mean'], s=80, color='C2', zorder=3)

texts = []
for _, row in merged.iterrows():
    texts.append(ax.text(row['arena_rating'], row['model_mean'], row['long_name'], fontsize=9))
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))

slope, intercept, r_value, p_value, _ = stats.linregress(merged['arena_rating'], merged['model_mean'])
x_range = np.linspace(merged['arena_rating'].min(), merged['arena_rating'].max(), 200)
ax.plot(x_range, slope * x_range + intercept, color='firebrick', linewidth=1.5, linestyle='--',
        label=f'r = {r_value:.2f},  p = {p_value:.3f}  (n={len(merged)})')

ax.axhline(0, color='gray', linewidth=2, linestyle='--')
ax.set_xlabel('LM Text Arena Rating', fontsize=13)
ax.set_ylabel('Political Bias — Model Mean Log Odds', fontsize=13)
ax.set_title(
    'Political Bias vs. Model Capability (LM Text Arena Rating)\n'
    'Non-political-domain person-attribution experiments only with subjective data',
    fontsize=14, fontweight='bold'
)
ax.legend(fontsize=15)
# ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(f'./figures/appendix_scatterplot_person_attribution_experiments_lm_arena_only_subjective_data.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:

import os
from adjustText import adjust_text

# Load ECI scores, drop models without a score
eci_df = pd.read_csv(os.path.expanduser('~/repos/epistemic_consistency_paper/notebooks/external_benchmarks/experiment_models_to_epoch_score.csv'))
eci_df = eci_df.dropna(subset=['eci'])

model_mean = (
    model_means_subjective.reset_index()
    .rename(columns={0: "model_mean"})
)
merged = eci_df.merge(model_mean, on='model_name', how='inner')

fig, ax = plt.subplots(figsize=(11, 7))

ax.scatter(merged['eci'], merged['model_mean'], s=80, color='C2', zorder=3)

texts = []
for _, row in merged.iterrows():
    texts.append(ax.text(row['eci'], row['model_mean'], row['long_name'], fontsize=9))
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))

slope, intercept, r_value, p_value, _ = stats.linregress(merged['eci'], merged['model_mean'])
x_range = np.linspace(merged['eci'].min(), merged['eci'].max(), 200)
ax.plot(x_range, slope * x_range + intercept, color='firebrick', linewidth=1.5, linestyle='--',
        label=f'r = {r_value:.2f},  p = {p_value:.3f}  (n={len(merged)})')

ax.axhline(0, color='gray', linewidth=2, linestyle='--')
ax.set_xlabel('ECI Score (Epoch)', fontsize=13)
ax.set_ylabel('Political Bias — Model Mean Log Odds', fontsize=13)
ax.set_title(
    'Political Bias vs. Model Capability (ECI)\n'
    'Non-political-domain person-attribution experiments only with subjective data',
    fontsize=14, fontweight='bold'
)
ax.legend(fontsize=15)
# ax.grid(True, alpha=0.3)
plt.tight_layout()
# fig.savefig(f'./figures/appendix_scatterplot_person_attribution_experiments_eci_score_only_subjective_data.png', dpi=300, bbox_inches='tight')
plt.show()
